# Standardize November's sources

*Data Analysis in Python · Topic 3.3*

The social team says November 2024's report undercounts Facebook and Instagram. Answer for November's export, the whole file.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the cell that reads the file into `november`.
2. In the first cell marked ✏️, create `raw_sources`: the number of different values in `session_source`. Look at them with `value_counts()`.
3. In the second cell marked ✏️:
   - Create `SOURCE_ALIASES`, a dictionary from each alias to its standard name, after lowercasing: `facebook.com`, `m.facebook.com`, `l.facebook.com`, `lm.facebook.com` and `fb` are `facebook`; `instagram.com`, `l.instagram.com` and `ig` are `instagram`.
   - Add a `source` column to `november`: `session_source` lowercased, stripped and with the aliases replaced. Leave `session_source` as it is.
4. In the third cell marked ✏️, create:
   - `clean_sources`: the number of different values in `source`
   - `facebook_sessions` and `instagram_sessions`: the total sessions whose `source` is `facebook` and `instagram`
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `raw_sources` is `22`. `november` has a `source` column, and its `session_source` column still has 22 spellings. `clean_sources` is `13`, `facebook_sessions` is `16833` and `instagram_sessions` is `10630`.

## Parameters

In [ ]:
import pandas as pd

FILE_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/sessions_export_2024_11.csv"

In [ ]:
november = pd.read_csv(FILE_URL, parse_dates=["date"])
november.shape

## The spellings

In [ ]:
raw_sources = november["session_source"].nunique()
november["session_source"].value_counts()

## One spelling per source

In [ ]:
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}
november["source"] = november["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sorted(november["source"].unique())

In [ ]:
clean_sources = november["source"].nunique()
facebook_sessions = november.loc[november["source"] == "facebook", "sessions"].sum()
instagram_sessions = november.loc[november["source"] == "instagram", "sessions"].sum()
clean_sources, facebook_sessions, instagram_sessions

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    raw_sources = found.get("raw_sources")
    if isinstance(raw_sources, (pd.DataFrame, pd.Series)):
        results.append((False, f"raw_sources is a {type(raw_sources).__name__}, not a number. Count the spellings: november[\"session_source\"].nunique()."))
    elif raw_sources == 22:
        results.append((True, "November's export has 22 spellings of its sources."))
    else:
        results.append((False, f"raw_sources should be november[\"session_source\"].nunique(), before any changes. It is {raw_sources!r}."))

    aliases = found.get("SOURCE_ALIASES")
    expected_aliases = {"facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook", "lm.facebook.com": "facebook", "fb": "facebook",
                        "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram"}
    if not isinstance(aliases, dict):
        results.append((False, "SOURCE_ALIASES should be a dictionary from alias to standard name, such as {\"fb\": \"facebook\", ...}."))
    elif missing := sorted(set(expected_aliases) - set(aliases)):
        results.append((False, f"SOURCE_ALIASES is missing {missing}. Every alias in the brief needs its own key, written in lowercase."))
    elif any(aliases[key] != value for key, value in expected_aliases.items()):
        results.append((False, "Some aliases in SOURCE_ALIASES point to the wrong name: the facebook spellings are \"facebook\", the instagram ones \"instagram\"."))
    else:
        results.append((True, "SOURCE_ALIASES has the eight aliases."))

    november = found.get("november")
    clean_sources = found.get("clean_sources")
    if not isinstance(november, pd.DataFrame) or len(november) != 2023:
        results.append((False, "november should be the whole November file, 2,023 rows."))
    elif "source" not in november.columns:
        results.append((False, "november has no source column yet. Add it: november[\"source\"] = november[\"session_source\"].str.lower().str.strip().replace(SOURCE_ALIASES)."))
    elif november["session_source"].nunique() != 22:
        results.append((False, "november's session_source column has changed. Put the standardized values in the new source column, and keep session_source as the export had it."))
    elif isinstance(clean_sources, (pd.DataFrame, pd.Series)):
        results.append((False, f"clean_sources is a {type(clean_sources).__name__}, not a number. Count the values: november[\"source\"].nunique()."))
    elif november["source"].nunique() == 13 and clean_sources == 13:
        results.append((True, "source has 13 values, one per real source, and session_source keeps the export's 22 spellings."))
    elif november["source"].nunique() == 14:
        results.append((False, "source has 14 values: Facebook with a capital F is still there. Lowercase before replacing the aliases: .str.lower().str.strip().replace(SOURCE_ALIASES)."))
    elif november["source"].nunique() == 21:
        results.append((False, "source has 21 values: it's lowercased, but the aliases aren't replaced. Add .replace(SOURCE_ALIASES)."))
    elif november["source"].nunique() == 13:
        results.append((False, f"source has the 13 values, but clean_sources is {clean_sources!r}. Count them with november[\"source\"].nunique()."))
    else:
        results.append((False, f"source has {november['source'].nunique()} values, but it should have 13. Lowercase, strip, then replace the aliases."))

    facebook = found.get("facebook_sessions")
    instagram = found.get("instagram_sessions")
    if tables := [name for name, value in [("facebook_sessions", facebook), ("instagram_sessions", instagram)] if isinstance(value, (pd.DataFrame, pd.Series))]:
        results.append((False, f"{' and '.join(tables)} should be numbers, not tables of rows. Total one column: november.loc[november[\"source\"] == \"facebook\", \"sessions\"].sum()."))
    elif facebook == 16833 and instagram == 10630:
        results.append((True, "Facebook had 16,833 sessions and Instagram 10,630, in November's export."))
    elif facebook == 10963:
        results.append((False, "facebook_sessions is 10,963: only the rows spelled exactly facebook in session_source. Total the rows whose new source column is facebook."))
    elif facebook == 15832:
        results.append((False, "facebook_sessions is 15,832: the rows spelled Facebook are missing. Lowercase before replacing the aliases."))
    else:
        results.append((False, f"facebook_sessions and instagram_sessions should be 16833 and 10630, but they are {facebook!r} and {instagram!r}. Total november.loc[november[\"source\"] == \"facebook\", \"sessions\"]."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
raw_sources = november["session_source"].nunique()
november["session_source"].value_counts()
```

```python
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}
november["source"] = november["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sorted(november["source"].unique())
```

```python
clean_sources = november["source"].nunique()
facebook_sessions = november.loc[november["source"] == "facebook", "sessions"].sum()
instagram_sessions = november.loc[november["source"] == "instagram", "sessions"].sum()
clean_sources, facebook_sessions, instagram_sessions
```

</details>